In [1]:
%%html
<script>
  function code_toggle() {
    if (code_shown){
      $('div.input').hide('500');
      $('#toggleButton').val('Show Code')
    } else {
      $('div.input').show('500');
      $('#toggleButton').val('Hide Code')
    }
    code_shown = !code_shown
  }

  $( document ).ready(function(){
    code_shown=false;
    $('div.input').hide()
  });
</script>
<form action="javascript:code_toggle()"><input type="submit" id="toggleButton" value="Show Code"></form>

In [ ]:
%%capture
%load_ext autoreload
%autoreload 2
%cd ..

# Tokenisation

### Choosing the units a model will process

## Tokenisation is a modelling decision

The string

> We can't re-use a word-level model in 北京.

could be represented as:

- words and punctuation;
- language-specific segments;
- characters or bytes;
- learned subwords.

The choice changes sequence length, vocabulary size and what the model can reuse.

## Whitespace is a useful baseline

It is fast and transparent—but spaces do not consistently mark linguistic units.

In [ ]:
text = "Mr. Bob Dobolina is thinkin' of a master plan. Why doesn't he quit?"
print(text.split())

['Mr.', 'Bob', 'Dobolina', 'is', "thinkin'", 'of', 'a', 'master', 'plan.', 'Why', "doesn't", 'he', 'quit?']

## A short regex can encode explicit conventions

- Keep **Mr.** together
- Keep apostrophes inside words
- Separate other punctuation

Regex is useful when the rules are known and local. It becomes brittle when languages and domains require many exceptions.

In [ ]:
import re

pattern = r"Mr\.|\w+(?:'\w+)*'?|[^\w\s]"
print(re.findall(pattern, text))

['Mr.', 'Bob', 'Dobolina', 'is', "thinkin'", 'of', 'a', 'master', 'plan', '.', 'Why', "doesn't", 'he', 'quit', '?']

## NLTK provides reusable tokenisation conventions

Use a tokenizer designed for the text you have:

- <code>word_tokenize</code> / <code>TreebankWordTokenizer</code> for conventional English text
- <code>TweetTokenizer</code> for handles, hashtags and emoticons
- <code>sent_tokenize</code> for sentence segmentation

Library tokenizers are strong baselines, not universal ground truth.

In [ ]:
from nltk.tokenize import TreebankWordTokenizer, TweetTokenizer

print(TreebankWordTokenizer().tokenize(text))
print(TweetTokenizer().tokenize("@diku Great course :-) #NLP"))

['Mr.', 'Bob', 'Dobolina', 'is', 'thinkin', "'", 'of', 'a', 'master', 'plan.', 'Why', 'does', "n't", 'he', 'quit', '?']
['@diku', 'Great', 'course', ':-)', '#NLP']

## No tokenizer is language- or domain-neutral

| Text | Boundary problem |
|---|---|
| 今日もしないといけない。 | Japanese normally has no spaces between words |
| thuế thu nhập cá nhân | Vietnamese spaces can separate syllables rather than words |
| (TPGS)-cisplatin | Biomedical punctuation may carry domain-specific structure |
| New York-based | Several reasonable tokenisations serve different tasks |

When fixed rules stop scaling, we can learn reusable units from data.

# Subword tokenisation

### A fixed vocabulary between words and characters

| Units | Vocabulary | Sequence length | Unseen strings |
|---|---:|---:|---|
| Words | large | short | problematic |
| Characters/bytes | small | long | always representable |
| Subwords | medium | medium | decomposed into known pieces |

## BPE repeatedly merges frequent adjacent symbols

Start with characters plus an end-of-word marker:

| Training word | Frequency | Initial symbols |
|---|---:|---|
| low | 5 | l · o · w · &lt;/w&gt; |
| lower | 2 | l · o · w · e · r · &lt;/w&gt; |
| newest | 6 | n · e · w · e · s · t · &lt;/w&gt; |
| widest | 3 | w · i · d · e · s · t · &lt;/w&gt; |

Count pairs **with corpus frequencies**, merge the most frequent pair, and repeat.

## Pair counts make the first merge concrete

In the weighted corpus:

- <code>e + s</code>: 6 occurrences in *newest* + 3 in *widest* = **9**
- <code>s + t</code>: 6 + 3 = **9**
- <code>l + o</code>: 5 in *low* + 2 in *lower* = **7**

Ties need a deterministic rule. Here we choose the alphabetically first pair.

## Count weighted adjacent pairs in Python

In [ ]:
from collections import Counter

corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}
vocab = {tuple(word) + ("</w>",): freq for word, freq in corpus.items()}

def pair_counts(vocabulary):
    counts = Counter()
    for symbols, frequency in vocabulary.items():
        for pair in zip(symbols, symbols[1:]):
            counts[pair] += frequency
    return counts

print(pair_counts(vocab).most_common(5))

[(('e', 's'), 9), (('s', 't'), 9), (('t', '</w>'), 9), (('w', 'e'), 8), (('l', 'o'), 7)]

## Replace every occurrence of the selected pair

In [ ]:
def merge_pair(vocabulary, pair):
    updated = {}
    for symbols, frequency in vocabulary.items():
        merged, i = [], 0
        while i < len(symbols):
            if i + 1 < len(symbols) and tuple(symbols[i:i + 2]) == pair:
                merged.append("".join(pair)); i += 2
            else:
                merged.append(symbols[i]); i += 1
        key = tuple(merged)
        updated[key] = updated.get(key, 0) + frequency
    return updated

## Repeat counting and merging

In [ ]:
def learn_bpe(vocabulary, number_of_merges):
    merges = []
    for step in range(number_of_merges):
        counts = pair_counts(vocabulary)
        best = min(counts, key=lambda pair: (-counts[pair], pair))
        print(f"{step + 1}: {best}  count={counts[best]}")
        merges.append(best)
        vocabulary = merge_pair(vocabulary, best)
    return merges

merges = learn_bpe(vocab, 5)

1: ('e', 's')  count=9
2: ('es', 't')  count=9
3: ('est', '</w>')  count=9
4: ('l', 'o')  count=7
5: ('lo', 'w')  count=7

## Encoding replays the learned merge order

Do **not** learn new merges from the test word. Start from its characters and apply the training merges in order.

In [ ]:
def encode_bpe(word, learned_merges):
    symbols = tuple(word) + ("</w>",)
    for pair in learned_merges:
        symbols = next(iter(merge_pair({symbols: 1}, pair)))
    return [s.replace("</w>", "") for s in symbols if s != "</w>"]

for word in ["lowest", "newer", "widest"]:
    print(f"{word:>6} -> {encode_bpe(word, merges)}")

lowest -> ['low', 'est']
 newer -> ['n', 'e', 'w', 'e', 'r']
widest -> ['w', 'i', 'd', 'est']

## Related algorithms make different vocabulary choices

| Method | Training idea | Familiar example |
|---|---|---|
| **BPE** | repeatedly merge frequent adjacent symbols | GPT-2 and RoBERTa use byte-level BPE |
| **WordPiece** | select pieces with a likelihood-inspired score | BERT; continuation pieces such as <code>##ing</code> |
| **Unigram** | start large and remove pieces that least improve the objective | T5 via SentencePiece |

**SentencePiece is a toolkit** that can train BPE or unigram models and represents whitespace explicitly.

# Sentence segmentation

Sentence boundaries are usually predicted rather than defined by one punctuation rule.

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize

sample = "Mr. Bob arrived at 9. It rained. Really?"
try:
    sentences = sent_tokenize(sample)
except LookupError:
    nltk.download("punkt_tab", quiet=True)
    sentences = sent_tokenize(sample)
print(sentences)

['Mr. Bob arrived at 9.', 'It rained.', 'Really?']

## Tokenisation checklist

- Start with a transparent baseline: whitespace, regex or an NLTK tokenizer
- Match tokenisation to language, domain and downstream labels
- For learned subwords, inspect the training objective and actual outputs
- Preserve offsets when tokens must map back to the original text
- Evaluate downstream errors—not tokenisation in isolation

Next: represent a document and predict its class.

# Background reading

- Jurafsky & Martin, [Chapter 2: Regular Expressions, Text Normalization, Edit Distance](https://web.stanford.edu/~jurafsky/slp3/2.pdf)
- NLTK, [Tokenization API](https://www.nltk.org/api/nltk.tokenize.html)
- Hugging Face NLP Course, [Tokenizers](https://huggingface.co/learn/nlp-course/chapter6/1)